# NYC Taxi Routes

## Project Facts

| Feld | Wert |
|------|------|
| **Business-Frage** | Wie viele Taxis sollte ein NYC-Taxiunternehmer am Flughafen JFK bereitstellen? |
| **Stakeholder** | Taxiunternehmer mit Flotte in New York City |
| **Methode** | EDA + Geo-Klassifikation (Pickup/Dropoff → JFK vs. NYC vs. Other) + Zeitreihen-Anteile |
| **Hauptdatenquelle** | `2016_Yellow_Taxi_prepared.csv` — 300.000 Taxifahrten, NYC 2016 |
| **Ziel-Metrik** | Anteil JFK-Fahrten an Gesamtfahrten (gesamt · pro Wochentag · pro Uhrzeit) |
| **Out of Scope** | Preismodellierung, Vorhersage-Modell — reine deskriptive Analyse |


## Project Context

### Scenario

Ein New Yorker Taxiunternehmer möchte die Anzahl seiner Taxis, welche am internationalen Flughafen JFK warten, optimieren.
Der Flughafen liegt deutlich abseits des Hauptgeschäftsgebiets Manhattan. Dadurch dauert es einerseits sehr lange, bis dort
wartende Taxis für andere Gebiete verfügbar werden. Andererseits legen am Flughafen ankommende Gäste in der Regel weitere
und somit lukrativere Strecken zurück.

### Mission

Analysiere, welchen Anteil JFK-Fahrten am Gesamtaufkommen haben — insgesamt sowie aufgeschlüsselt nach Wochentag und
Uhrzeit — und leite daraus eine Empfehlung ab, wie viele Taxis der Unternehmer am Flughafen bereitstellen sollte.

### Scope / Hypotheses

* JFK-Fahrten sind ein kleiner, aber überproportional lukrativer Anteil des Geschäfts (weitere Strecken, höhere Fahrpreise)
* Der JFK-Anteil schwankt vermutlich systematisch nach Wochentag und Uhrzeit (Geschäftsreisen vs. Freizeit)
* Herkunft: [StackFuel](https://stackfuel.com) Übungsprojekt, Modul 2 / Kapitel 7 — Original-Aufgabenstellung in [`docs/infos.md`](../docs/infos.md)


### Methode & Metrics

| Metrik | Zielwert | Begruendung |
|--------|----------|-------------|
| Anteil JFK-Fahrten (gesamt) | Referenzwert für Flottenplanung | Kern-Fragestellung des Auftraggebers |
| Anteil JFK-Fahrten pro Wochentag | Höchster/niedrigster Tag identifizieren | Steuert Flottenverteilung über die Woche |
| Anteil JFK-Fahrten pro Uhrzeit | Peak-Stunden identifizieren | Steuert Schichtplanung |


### Data Dictionary

| Spaltennummer | Spaltenname | Datenniveau | Beschreibung |
| :--- | :--- | :--- | :--- |
| 0 | `'pickup_weekday'` | kategorisch (ordinal) | Wochentag, an dem die Fahrt begonnen hat (0=Montag, 6=Sonntag) |
| 1 | `'pickup_hour'` | kategorisch (ordinal) | Stunde, in der die Fahrt begonnen hat |
| 2 | `'pickup_longitude'` | numerisch (`float`) | Längengrad, bei dem die Fahrt begonnen hat |
| 3 | `'pickup_latitude'` | numerisch (`float`) | Breitengrad, bei dem die Fahrt begonnen hat |
| 4 | `'dropoff_longitude'` | numerisch (`float`) | Längengrad, bei dem die Fahrt geendet hat |
| 5 | `'dropoff_latitude'` | numerisch (`float`) | Breitengrad, bei dem die Fahrt geendet hat |
| 6 | `'passenger_count'` | kategorisch (ordinal) | Anzahl der Passagiere im Auto (manuell erfasst) |
| 7 | `'trip_distance'` | numerisch (`float`) | Zurückgelegte Fahrtstrecke in Meilen |
| 8 | `'fare_amount'` | numerisch (`float`) | Taxameter-Betrag basierend auf Zeit und Strecke |
| 9 | `'tip_amount'` | numerisch (`float`) | Trinkgeld bei Kartenzahlung (0.00 bei Barzahlung) |
| 10 | `'tolls_amount'` | numerisch (`float`) | Angefallene Maut-Gebühren |
| 11 | `'payment_type'` | kategorisch (nominal) | Zahlungsart (1=Kreditkarte, 2=Bar, 3=keine Gebühr, 4=Streitigkeit) |

**Geo-Referenzwerte** (siehe `nyc_taxi_routes.utils.JFK` / `.NYC`):

| Bounds | lat_min | lat_max | lon_min | lon_max |
| :--- | :---: | :---: | :---: | :---: |
| JFK | 40.62666 | 40.66018 | -73.80822 | -73.76599 |
| NYC | 40.5774 | 40.9176 | -74.15 | -73.7004 |


## Workflow

### Phases

| Phase | 01 Exploration & Discovery | 02 Preparation & Preprocessing | 03 Analysis & Analytics | 04 Communication & Insights |
|-------|-------------------|---------------------|------------------|------------------|
| | Data Acquisition  | Data Cleaning       | Statistical Analysis | Insight Delivery |
| | Initial Profiling | Outlier Handling    | Pattern Recognition  | Data Storytelling |
| | Quality Audit     | Transformation      | Hypothesis Testing   | Strategic Advice |
| | Key Findings      | Feature Engineering | Result Aggregation   | Executive Summary|

### Conventions

| Variable | Phase | Zustand |
|----------|-------|---------|
| `df_raw` | Loading | Originalzustand – schreibgeschützt |
| `df_eda` | Exploration | Initiale Daten zur explorativen Analyse |
| `df_edit` | Preparation & Processing | Bereinigt, transformiert & aggregiert (Arbeitsbasis) |
| `df_final` | Analysis & Reporting | Finaler Output für Insights und Visualisierungen |

> **Best Practice:** Nutze für jeden Transformationsschritt `.copy()`, um die Traceability zu gewährleisten und `df_raw` niemals zu überschreiben.